In [1]:
import pandas as pd
import pyarrow.parquet as pq

# Set display option to avoid scientific notation
pd.set_option('display.float_format', '{:.10f}'.format)

pq_path = r"parquet_files_dataset/compress/compress-gzip-all-events-run0.parquet"
start, stop = 1, 20

pf = pq.ParquetFile(pq_path)
tbl = pf.read()
df = tbl.to_pandas().iloc[start:stop]

display(df)

,t_sec,dt_sec,cpu,event_id
1,31591.4913498300,0.0000005360,1,33
2,31591.4913506380,0.0000008080,1,34
3,31591.4913512700,0.0000006320,2,23
4,31591.4913517800,0.0000005100,1,20
5,31591.4913521580,0.0000003780,2,18
6,31591.4913524730,0.0000003150,2,18
7,31591.4913527640,0.0000002910,2,25
8,31591.4913528860,0.0000001220,1,19
9,31591.4913533560,0.0000004700,1,20
10,31591.4913536500,0.0000002940,1,19


In [3]:
import numpy as np

def inspect_npz(npz_path, n=3, k=15):
    d = np.load(npz_path)
    event = d["event"]
    dt    = d["dt"]
    cpu   = d["cpu"]

    print("=== SHAPES ===")
    print("event:", event.shape, event.dtype)
    print("dt   :", dt.shape, dt.dtype)
    print("cpu  :", cpu.shape, cpu.dtype)

    B, L = event.shape

    print("\n=== SAMPLE SEQUENCES (first 20 tokens) ===")
    for i in range(min(n, B)):
        print(f"\n-- seq {i} --")
        print("event:", event[i, :20].tolist())
        print("dt   :", dt[i, :20].tolist())
        print("cpu  :", cpu[i, :20].tolist())

    print("\n=== BASIC STATS ===")
    print("event min/max:", int(event.min()), int(event.max()))
    print("dt    min/max:", int(dt.min()), int(dt.max()))
    print("cpu   min/max:", int(cpu.min()), int(cpu.max()))

    ev_counts = np.bincount(event.reshape(-1))
    top = np.argsort(ev_counts)[::-1][:k]

    print(f"\nTop {k} events:")
    for eid in top:
        if ev_counts[eid] == 0:
            break
        print(f"  event_id={eid}: {int(ev_counts[eid])}")

    cpu_counts = np.bincount(cpu.reshape(-1))
    print("\nCPU distribution:")
    for c in range(len(cpu_counts)):
        if cpu_counts[c] > 0:
            print(f"  cpu {c}: {int(cpu_counts[c])}")

    dt_flat = dt.reshape(-1).astype(np.int64)
    print("\nDT bucket stats:")
    print(
        "  min=", int(dt_flat.min()),
        "max=", int(dt_flat.max()),
        "mean=", float(dt_flat.mean()),
    )

In [4]:
npz_path = r"window_shards\compress-gzip\train\run00_shard0000.npz"
inspect_npz(npz_path, n=5, k=20)

=== SHAPES ===
event: (100000, 200) int32
dt   : (100000, 200) uint8
cpu  : (100000, 200) uint8

=== SAMPLE SEQUENCES (first 20 tokens) ===

-- seq 0 --
event: [19, 33, 34, 23, 20, 18, 18, 25, 19, 20, 19, 117, 1, 55, 8, 8, 95, 23, 23, 115]
dt   : [92, 73, 77, 75, 72, 69, 66, 66, 55, 71, 66, 79, 49, 66, 68, 71, 80, 96, 53, 57]
cpu  : [1, 1, 1, 2, 1, 2, 2, 2, 1, 1, 1, 2, 1, 1, 1, 1, 2, 0, 3, 1]

-- seq 1 --
event: [19, 11, 20, 7, 19, 14, 20, 19, 20, 1, 19, 20, 8, 19, 8, 20, 20, 8, 19, 19]
dt   : [70, 56, 69, 69, 49, 54, 62, 62, 73, 46, 71, 64, 45, 60, 70, 69, 56, 71, 59, 61]
cpu  : [1, 2, 1, 2, 1, 2, 1, 1, 1, 2, 1, 1, 2, 1, 2, 1, 2, 2, 1, 2]

-- seq 2 --
event: [20, 19, 8, 20, 19, 8, 20, 19, 20, 6, 19, 0, 6, 20, 0, 6, 0, 6, 0, 19]
dt   : [53, 72, 58, 44, 62, 59, 64, 73, 70, 71, 36, 57, 61, 53, 35, 59, 54, 56, 54, 35]
cpu  : [1, 1, 2, 1, 1, 2, 1, 1, 1, 2, 1, 2, 2, 1, 2, 2, 2, 2, 2, 1]

-- seq 3 --
event: [20, 10, 13, 19, 11, 14, 20, 19, 19, 7, 20, 8, 19, 20, 158, 158, 8, 8, 11, 7]
dt   : 

In [5]:
npz_path = r"window_shards\compress-gzip\train\run00_shard0004.npz"
inspect_npz(npz_path, n=5, k=20)

=== SHAPES ===
event: (99771, 200) int32
dt   : (99771, 200) uint8
cpu  : (99771, 200) uint8

=== SAMPLE SEQUENCES (first 20 tokens) ===

-- seq 0 --
event: [1, 1, 1, 1, 18, 18, 24, 25, 21, 22, 23, 17, 0, 0, 0, 0, 0, 0, 0, 33]
dt   : [71, 73, 71, 70, 71, 53, 56, 63, 67, 60, 69, 139, 78, 70, 69, 69, 69, 70, 70, 61]
cpu  : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]

-- seq 1 --
event: [33, 34, 16, 23, 21, 22, 18, 18, 25, 1, 1, 1, 1, 1, 1, 1, 3, 17, 16, 2]
dt   : [61, 71, 63, 53, 63, 61, 63, 53, 61, 77, 71, 72, 71, 71, 70, 71, 73, 67, 110, 62]
cpu  : [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

-- seq 2 --
event: [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 33, 34, 16, 23]
dt   : [62, 61, 66, 61, 61, 61, 61, 61, 60, 56, 68, 68, 68, 69, 68, 68, 62, 71, 64, 48]
cpu  : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]

-- seq 3 --
event: [5, 5, 5, 5, 5, 5, 5, 1, 1, 1, 1, 1, 1, 18, 18, 24, 25, 21, 22, 23]
dt   : [58, 58, 58, 58, 59, 58, 58, 59